In [2]:
import os
import json
import warnings
import numpy as np
import xarray as xr
import proplot as pplt
warnings.filterwarnings('ignore')
pplt.rc.update({
    'tick.minor':False,
    'savefig.dpi':300,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'legend.fontsize':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

In [3]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR = CONFIGS['filepaths']['splits']
PREDSDIR  = CONFIGS['filepaths']['predictions']
LATRANGE  = CONFIGS['domain']['latrange']
LONRANGE  = CONFIGS['domain']['lonrange']
SPLIT     = 'valid'
_MODELS   = CONFIGS['experiments']
MODELS    = {}
for _name,_rc in _MODELS['pod']['runs'].items():
    MODELS[_name] = _rc.get('description',_name)
for _name,_rc in _MODELS['nn']['runs'].items():
    MODELS[_name] = _rc.get('description',_name)
for _name,_eqspec in _MODELS['sr'].get('optimizedeqs',{}).items():
    MODELS[_name] = _eqspec.get('description',_name)

In [4]:
with xr.open_dataset(os.path.join(SPLITSDIR,f'{SPLIT}.h5'),engine='h5netcdf') as ds:
    truetp = ds.tp.load()

results = {}
for name in MODELS.keys():
    filepath = os.path.join(PREDSDIR,f'{name}_{SPLIT}_predictions.nc')
    if not os.path.exists(filepath):
        continue
    with xr.open_dataset(filepath) as ds:
        pred = ds.tp.load()
    if 'seed' in pred.dims:
        pred = pred.mean('seed')
    if 'complexity' in pred.dims:
        pred = pred.isel(complexity=0)
    ytrue,ypred   = xr.align(truetp,pred,join='inner')
    results[name] = (ytrue.squeeze(),ypred.squeeze())

print(f'Loaded {len(results)}/{len(MODELS)} models!')

Loaded 8/8 models!


In [ ]:
def spatial_r2(ytrue,ypred):
    ssres = ((ytrue-ypred)**2).sum('time',skipna=True)
    sstot = ((ytrue-ytrue.mean('time',skipna=True))**2).sum('time',skipna=True)
    return (1-ssres/sstot).squeeze()

names = sorted([n for n in MODELS.keys() if n in results],
               key=lambda n: float(spatial_r2(*results[n]).mean()))
# Ensure POD-BL always precedes SR-LO regardless of their relative R² values
if 'pod_bl' in names and 'sr_lo' in names:
    i_pod,i_sr = names.index('pod_bl'),names.index('sr_lo')
    if i_sr < i_pod:
        names.remove('sr_lo')
        names.insert(names.index('pod_bl') + 1, 'sr_lo')
n     = len(names)
ncols = 4
nrows = int(np.ceil(n/ncols))

fig,axs = pplt.subplots(nrows=nrows,ncols=ncols,proj='cyl',figwidth=6.5,share=False,wspace=0)
axs.format(coast=True,borders=False,latlim=LATRANGE,lonlim=LONRANGE,latlines=5,lonlines=5,grid=False)
m = None
for ax,name in zip(axs,names):
    ytrue,ypred = results[name]
    r2          = spatial_r2(ytrue,ypred)
    m = ax.pcolormesh(r2.lon,r2.lat,r2,cmap='ColdHot',cmap_kw={'left':0.5},vmin=0,vmax=1,levels=11,extend='min')
    if m is None:
        m = m
    ax.format(title=MODELS.get(name,name))
for ax in axs[n:]:
    ax.set_visible(False)
fig.colorbar(m,loc='b',label='Time-Mean R$^2$',ticks=0.1)
pplt.show()
fig.save('../figs/fig_3.jpg')